# 06 — MLflow Experiment Tracking
**Dataset:** Fruits 360  
**Tujuan Notebook:** Log semua eksperimen, parameter, metrik, dan model artifact ke MLflow.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import pickle
import mlflow
import mlflow.sklearn
import os

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = 'data/processed'
MODEL_DIR  = 'models'
print('Libraries loaded!')

## 2. Setup MLflow

In [ ]:
# Set experiment name
mlflow.set_experiment('fruits360-image-classification')

# URI default: folder ./mlruns di direktori aktif
print(f'MLflow Tracking URI: {mlflow.get_tracking_uri()}')
print('Experiment siap!')

## 3. Load Data & Scaler

In [ ]:
X_train = np.load(f'{OUTPUT_DIR}/X_train_pca.npy')
X_test  = np.load(f'{OUTPUT_DIR}/X_test_pca.npy')
y_train = np.load(f'{OUTPUT_DIR}/y_train.npy')
y_test  = np.load(f'{OUTPUT_DIR}/y_test.npy')

with open(f'{MODEL_DIR}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Data siap!')

## 4. Helper Function untuk Logging

In [ ]:
def log_model_run(model, model_name, params, X_train, y_train, X_test, y_test):
    """
    Melatih model dan mencatat semua parameter, metrik, dan artifact ke MLflow.
    """
    with mlflow.start_run(run_name=model_name):
        # Log parameter
        mlflow.log_params(params)
        mlflow.log_param('model_type', model_name)
        mlflow.log_param('n_train_samples', len(X_train))
        mlflow.log_param('n_test_samples', len(X_test))
        mlflow.log_param('n_features', X_train.shape[1])

        # Training
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Hitung metrik
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # Log metrik
        mlflow.log_metric('accuracy',  acc)
        mlflow.log_metric('precision', prec)
        mlflow.log_metric('recall',    rec)
        mlflow.log_metric('f1_score',  f1)

        # Log model artifact
        mlflow.sklearn.log_model(model, artifact_path='model')

        print(f'[{model_name}] Acc={acc:.4f} | F1={f1:.4f} | Run logged!')
        return acc, f1

## 5. Log Semua Model

In [ ]:
experiments = [
    {
        'model': SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
        'name': 'SVM_RBF_baseline',
        'params': {'kernel': 'rbf', 'C': 10, 'gamma': 'scale'}
    },
    {
        'model': SVC(kernel='linear', C=1.0, random_state=42),
        'name': 'SVM_Linear',
        'params': {'kernel': 'linear', 'C': 1.0}
    },
    {
        'model': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'name': 'RandomForest_100trees',
        'params': {'n_estimators': 100, 'max_depth': 'None'}
    },
    {
        'model': RandomForestClassifier(n_estimators=200, max_depth=30, random_state=42, n_jobs=-1),
        'name': 'RandomForest_200trees_depth30',
        'params': {'n_estimators': 200, 'max_depth': 30}
    },
    {
        'model': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'name': 'KNN_k5',
        'params': {'n_neighbors': 5, 'metric': 'euclidean'}
    },
    {
        'model': KNeighborsClassifier(n_neighbors=10, n_jobs=-1),
        'name': 'KNN_k10',
        'params': {'n_neighbors': 10, 'metric': 'euclidean'}
    },
    {
        'model': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
        'name': 'LogisticRegression_C1',
        'params': {'C': 1.0, 'max_iter': 1000}
    },
]

print('Mulai logging semua eksperimen ke MLflow...')
all_results = []
for exp in experiments:
    acc, f1 = log_model_run(
        exp['model'], exp['name'], exp['params'],
        X_train_scaled, y_train,
        X_test_scaled,  y_test
    )
    all_results.append({'Run': exp['name'], 'Accuracy': acc, 'F1-Score': f1})

results_df = pd.DataFrame(all_results).sort_values('F1-Score', ascending=False)
print('\n=== Semua Run Selesai ===')
print(results_df.to_string(index=False))

## 6. Log Model Terbaik (dari Optuna)

In [ ]:
with open(f'{MODEL_DIR}/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

with mlflow.start_run(run_name='BEST_MODEL_Optuna_Tuned'):
    mlflow.log_param('model_type', str(type(best_model).__name__))
    mlflow.log_param('tuning_method', 'Optuna')
    mlflow.log_param('n_trials', 30)
    mlflow.log_params(best_model.get_params())

    y_pred = best_model.predict(X_test_scaled)
    mlflow.log_metric('accuracy',  accuracy_score(y_test, y_pred))
    mlflow.log_metric('precision', precision_score(y_test, y_pred, average='weighted', zero_division=0))
    mlflow.log_metric('recall',    recall_score(y_test, y_pred, average='weighted', zero_division=0))
    mlflow.log_metric('f1_score',  f1_score(y_test, y_pred, average='weighted', zero_division=0))

    mlflow.sklearn.log_model(best_model, artifact_path='best_model')

    print(f'Best Model logged! Accuracy: {accuracy_score(y_test, y_pred):.4f}')

## 7. Cara Membuka MLflow UI

In [ ]:
print('='*50)
print('Untuk membuka MLflow UI, jalankan di terminal:')
print()
print('    mlflow ui')
print()
print('Lalu buka browser ke: http://localhost:5000')
print('='*50)
print()
print(f'Total run yang tercatat: {len(experiments) + 1}')
print('Experiment name: fruits360-image-classification')

## 8. Ringkasan
- Semua eksperimen (7 run baseline + 1 run Optuna tuned) berhasil di-log ke MLflow
- Setiap run mencatat: parameter, accuracy, precision, recall, F1-score, dan model artifact
- Gunakan `mlflow ui` untuk visualisasi dan perbandingan antar run secara interaktif
- **Proyek selesai!** Semua notebook sudah berjalan end-to-end ✅